# M3L3 E23 - Multiagente simple con orquestador + RAG basico

Este ejercicio es una base simple para explicar sistemas multiagente sin sumar Langfuse, evaluadores automaticos ni configuracion avanzada.

Objetivo de la clase:
- Entender para que sirve un orquestador.
- Entender para que sirven los agentes especializados.
- Entender que significa hacer un RAG simple antes de responder.
- Ver como LangGraph conecta todo en un flujo claro.

Idea central:

```text
Usuario pregunta
      |
      v
orquestador decide area
      |
      v
agente especialista recupera contexto simple (RAG)
      |
      v
LLM responde usando solo ese contexto
```

Este ejercicio es deliberadamente simple. Despues se puede mejorar con embeddings, vector stores, memoria, herramientas o Langfuse.

## Paso 1 - Instalar dependencias y crear el modelo

Vamos a usar solo dos piezas externas:

- `langgraph`: para armar el flujo como grafo.
- `langchain-openai`: para llamar a OpenAI desde LangChain.

No usamos Langfuse en este ejercicio. Queremos que el foco sea el diseno del sistema: orquestador, agentes y RAG basico.

`temperature=0` hace que el modelo sea mas estable para clase.

In [ ]:
# Instalamos solo lo necesario para este ejercicio:
# - langgraph: permite crear el grafo de nodos.
# - langchain-openai: permite usar ChatOpenAI con una interfaz simple.
!pip install langgraph langchain-openai -q

# `os` permite guardar la API key como variable de entorno.
# `re` se usa despues para separar palabras en el retriever simple.
# `getpass` oculta la API key en la salida del notebook.
# `TypedDict` nos ayuda a documentar la forma del estado compartido.
import os
import re
from getpass import getpass
from typing import TypedDict

# ChatOpenAI es el wrapper de LangChain para llamar al modelo de OpenAI.
from langchain_openai import ChatOpenAI

# StateGraph arma el flujo. START y END marcan inicio y fin.
from langgraph.graph import StateGraph, START, END

# La API key se pide en runtime para no dejar secretos escritos en el notebook.
os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ").strip()

# Un solo modelo para todos los agentes.
# En sistemas reales podriamos usar modelos distintos segun costo, velocidad o precision.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("Setup listo: LangGraph + OpenAI")

## Paso 2 - Definir el estado compartido

En LangGraph los nodos se comunican usando un estado.

El estado es como la carpeta del caso: cada nodo lee algunos campos y escribe otros.

Campos:
- `query`: pregunta original del usuario.
- `agent`: agente elegido por el orquestador.
- `context`: documentos recuperados por RAG simple.
- `answer`: respuesta final generada por el agente.

Este diseno nos permite explicar cada paso: entrada, decision, contexto y respuesta.

In [ ]:
# SupportState define el contrato del flujo.
# No ejecuta nada por si solo: solo documenta que campos viajan por el grafo.
class SupportState(TypedDict):
    # Pregunta original del usuario.
    query: str

    # Nombre del agente elegido por el orquestador: sales, tech, billing o unknown.
    agent: str

    # Contexto recuperado por el agente antes de llamar al LLM.
    context: str

    # Respuesta final que vera el usuario.
    answer: str

print("State definido")

## Paso 3 - Base de conocimiento simple

Para hacer RAG necesitamos documentos. En produccion podrian venir de PDFs, Notion, una wiki, una base vectorial o una base de datos.

Aca usamos listas de texto para que sea facil de ver.

Tenemos tres areas:
- `sales`: planes, precios y demos.
- `tech`: errores tecnicos, API, webhooks e integraciones.
- `billing`: facturacion, pagos, reembolsos e impuestos.

Este RAG todavia no usa embeddings. Recupera documentos por palabras compartidas con la consulta. Es simple, pero alcanza para explicar la idea.

In [ ]:
# DOCS simula una base de conocimiento interna.
# Cada key es un area de negocio y cada lista contiene documentos cortos.
DOCS = {
    "sales": [
        "Plan Starter: hasta 5 usuarios, soporte por email y funcionalidades basicas.",
        "Plan Pro: hasta 50 usuarios, automatizaciones, reportes y soporte prioritario.",
        "Plan Enterprise: usuarios ilimitados, SSO, auditoria avanzada y manager dedicado.",
        "Las demos comerciales se coordinan con el equipo de ventas y duran 30 minutos.",
    ],
    "tech": [
        "API: todas las requests deben incluir Authorization: Bearer <token>.",
        "Webhooks: si no llegan eventos, revisar URL publica, HTTPS y respuesta 2xx.",
        "Integraciones: Slack, GitHub y Jira estan disponibles en el plan Pro o superior.",
        "Errores 401 suelen indicar token invalido, vencido o faltante.",
    ],
    "billing": [
        "Facturacion: las facturas se emiten el primer dia habil de cada mes.",
        "Metodo de pago: se puede actualizar desde Settings > Billing.",
        "Reembolsos: se aceptan dentro de los 15 dias posteriores al pago.",
        "Impuestos: la factura incluye impuestos segun pais de residencia fiscal.",
    ],
}


def tokenize(text: str) -> set[str]:
    # Tokenizador minimo para clase.
    # Convierte a minusculas y conserva palabras de 3 o mas caracteres.
    # Ejemplo: "Mi webhook falla" -> {"webhook", "falla"}
    return set(re.findall(r"[a-z0-9]{3,}", text.lower()))


def retrieve_docs(area: str, query: str, k: int = 2) -> list[str]:
    # Este es el RAG simple del ejercicio.
    # 1. Tokenizamos la consulta.
    # 2. Tokenizamos cada documento del area.
    # 3. Calculamos cuantas palabras comparten.
    # 4. Devolvemos los k documentos con mayor score.
    query_tokens = tokenize(query)
    scored = []

    for doc in DOCS.get(area, []):
        doc_tokens = tokenize(doc)
        score = len(query_tokens & doc_tokens)
        scored.append((score, doc))

    # Orden descendente: primero documentos con mas palabras compartidas.
    scored.sort(key=lambda item: item[0], reverse=True)

    # Devolvemos solo los documentos, no los scores, porque eso es lo que ira al prompt.
    return [doc for score, doc in scored[:k]]

print("Documentos y retriever simple listos")

## Paso 4 - Orquestador

El orquestador no responde la pregunta final. Su trabajo es decidir quien debe responder.

En este ejercicio usamos reglas por palabras clave para que la clase sea estable y facil de seguir.

Regla mental:
- Si la pregunta habla de precios, planes o demos: `sales`.
- Si habla de API, errores, webhooks o integraciones: `tech`.
- Si habla de facturas, pagos o reembolsos: `billing`.
- Si no queda claro: `unknown`.

Mas adelante podriamos reemplazar estas reglas por un clasificador con LLM.

In [ ]:
def route_query(query: str) -> str:
    # Normalizamos a minusculas para comparar de forma simple.
    q = query.lower()

    # Cada lista representa senales de una categoria.
    # No es perfecto, pero es transparente para clase.
    sales_words = ["precio", "plan", "enterprise", "starter", "pro", "demo", "usuarios"]
    tech_words = ["api", "webhook", "error", "token", "401", "integracion", "slack", "github", "jira"]
    billing_words = ["factura", "pago", "billing", "reembolso", "impuesto", "tarjeta", "metodo"]

    # El primer match decide el area.
    if any(word in q for word in sales_words):
        return "sales"
    if any(word in q for word in tech_words):
        return "tech"
    if any(word in q for word in billing_words):
        return "billing"

    # Si no hay senales claras, lo mandamos a fallback.
    return "unknown"


def orchestrator_node(state: SupportState) -> dict:
    # Este es el nodo orquestador.
    # Lee `query`, decide `agent` y devuelve solo ese campo actualizado.
    # No llama al LLM y no genera la respuesta final.
    agent = route_query(state["query"])
    return {"agent": agent}

print("Orquestador definido")

## Paso 5 - Agentes especializados con RAG simple

Cada agente hace tres cosas:

1. Recibe la consulta original.
2. Recupera documentos de su area con `retrieve_docs(...)`.
3. Le pide al LLM que responda usando solo ese contexto.

Esto es RAG en version minima:

```text
consulta + documentos recuperados -> prompt -> LLM -> respuesta
```

No estamos entrenando el modelo. Le estamos dando contexto en el prompt.

In [ ]:
def answer_with_context(area: str, state: SupportState) -> dict:
    # Esta funcion contiene la logica comun de todos los agentes.
    # Asi evitamos duplicar codigo en sales_agent, tech_agent y billing_agent.

    # Paso 1: recuperar documentos del area elegida por el orquestador.
    docs = retrieve_docs(area, state["query"], k=2)

    # Paso 2: convertir la lista de documentos en texto para el prompt.
    context = "\n".join(f"- {doc}" for doc in docs)

    # Paso 3: construir un prompt que obligue al modelo a usar el contexto.
    # Esto es importante: el LLM no deberia inventar fuera de los documentos.
    prompt = (
        f"Sos un agente especialista del area {area}.\n"
        "Responde la consulta usando unicamente el contexto provisto.\n"
        "Si el contexto no alcanza, deci que dato falta sin inventar.\n\n"
        f"Contexto recuperado:\n{context}\n\n"
        f"Consulta del usuario: {state['query']}\n\n"
        "Respuesta breve, clara y en espanol:"
    )

    # Paso 4: llamar al modelo.
    response = llm.invoke(prompt)

    # Paso 5: devolver solo los campos nuevos del estado.
    return {
        "context": context,
        "answer": response.content.strip(),
    }


def sales_agent(state: SupportState) -> dict:
    # Agente comercial: planes, precios, demos y usuarios.
    return answer_with_context("sales", state)


def tech_agent(state: SupportState) -> dict:
    # Agente tecnico: API, webhooks, tokens, errores e integraciones.
    return answer_with_context("tech", state)


def billing_agent(state: SupportState) -> dict:
    # Agente de facturacion: pagos, facturas, reembolsos e impuestos.
    return answer_with_context("billing", state)


def fallback_agent(state: SupportState) -> dict:
    # Este agente no llama al LLM.
    # Responde cuando el orquestador no encontro un area clara.
    return {
        "context": "",
        "answer": "No tengo suficiente informacion para derivar esta consulta. Proba preguntando por planes, API, webhooks, facturacion o pagos.",
    }


def choose_agent(state: SupportState) -> str:
    # Esta funcion se usa en add_conditional_edges.
    # LangGraph espera que devolvamos el nombre del proximo nodo.
    return {
        "sales": "sales_agent",
        "tech": "tech_agent",
        "billing": "billing_agent",
    }.get(state["agent"], "fallback_agent")

print("Agentes definidos")


## Paso 6 - Armar el grafo

Ahora conectamos todo.

El flujo completo queda asi:

```text
START
  |
  v
orchestrator_node
  |--------- sales_agent
  |--------- tech_agent
  |--------- billing_agent
  |--------- fallback_agent
              |
              v
             END
```

La parte importante es `add_conditional_edges`: ahi LangGraph decide el proximo nodo segun el valor de `state["agent"]`.

In [ ]:
# Creamos un grafo cuyo estado tiene la forma SupportState.
graph = StateGraph(SupportState)

# Registramos todos los nodos disponibles.
# El nombre string es lo que LangGraph usa internamente para conectar el flujo.
graph.add_node("orchestrator_node", orchestrator_node)
graph.add_node("sales_agent", sales_agent)
graph.add_node("tech_agent", tech_agent)
graph.add_node("billing_agent", billing_agent)
graph.add_node("fallback_agent", fallback_agent)

# El flujo siempre empieza por el orquestador.
graph.add_edge(START, "orchestrator_node")

# Despues del orquestador, LangGraph llama a choose_agent(state).
# Esa funcion devuelve el nombre del nodo que debe ejecutarse.
graph.add_conditional_edges(
    "orchestrator_node",
    choose_agent,
    {
        "sales_agent": "sales_agent",
        "tech_agent": "tech_agent",
        "billing_agent": "billing_agent",
        "fallback_agent": "fallback_agent",
    },
)

# Cualquier agente termina el flujo.
for node in ["sales_agent", "tech_agent", "billing_agent", "fallback_agent"]:
    graph.add_edge(node, END)

# compile valida y prepara el grafo para ejecutar con app.invoke(...).
app = graph.compile()
print("Grafo compilado")

## Paso 7 - Ejecutar consultas de prueba

Cada consulta muestra cuatro cosas:

- La pregunta original.
- Que agente eligio el orquestador.
- Que contexto recupero el agente.
- Que respuesta genero el LLM.

Frase para clase:

> El orquestador decide la ruta. El agente trae contexto de su area. El LLM no responde desde la nada: responde apoyado en documentos recuperados.

In [ ]:
def run_query(query: str):
    # Estado inicial: solo tenemos la pregunta del usuario.
    # Los otros campos empiezan vacios y los nodos los van completando.
    initial_state = {
        "query": query,
        "agent": "",
        "context": "",
        "answer": "",
    }

    # Ejecutamos el grafo completo.
    result = app.invoke(initial_state)

    # Imprimimos cada parte para explicar el flujo en clase.
    print("=" * 80)
    print("Consulta:      ", query)
    print("Agente elegido:", result["agent"])
    print()
    print("Contexto RAG:")
    print(result["context"] or "(sin contexto)")
    print()
    print("Respuesta:")
    print(result["answer"])
    return result


# Una consulta para cada ruta principal y una fuera de alcance.
queries = [
    "Que incluye el plan Enterprise",
    "Mi webhook no esta llegando al servidor",
    "Necesito cambiar el metodo de pago",
    "Cual es la capital de Francia",
]

for query in queries:
    run_query(query)


## Paso 8 - Mirar solo el RAG sin LLM

Esta celda sirve para explicar retrieval sin gastar tokens.

Antes de que el LLM responda, el agente busca documentos. Podemos probar esa busqueda directamente.

Esto ayuda a separar dos conceptos:
- Retrieval: encontrar contexto relevante.
- Generation: redactar una respuesta con el LLM.

In [ ]:
# Esta prueba no llama al LLM.
# Solo muestra que documentos recuperaria el sistema para una consulta tecnica.
query = "webhook error token"

# Primero usamos el mismo router que usa el grafo.
area = route_query(query)

# Despues recuperamos documentos del area elegida.
docs = retrieve_docs(area, query, k=2)

print("Consulta:", query)
print("Area elegida:", area)
print("Documentos recuperados:")
for doc in docs:
    print("-", doc)

## Cierre - Que aprendiste

En este ejercicio construiste una base simple de sistema multiagente:

| Concepto | Que hace en este notebook |
|---|---|
| Orquestador | Decide que agente debe responder |
| Agente especialista | Responde solo desde su area |
| RAG simple | Recupera documentos relevantes antes de llamar al LLM |
| LangGraph | Conecta nodos y rutas condicionales |
| Fallback | Maneja consultas fuera de alcance |

Frase clave:

> Un sistema multiagente no es muchos chats sueltos. Es un flujo donde cada agente tiene una responsabilidad concreta y el orquestador decide cuando usarlo.

Proximas mejoras posibles:
- Cambiar el router por un clasificador con LLM.
- Reemplazar `retrieve_docs` por embeddings + vector store.
- Agregar memoria de conversacion.
- Agregar observabilidad con Langfuse en otro ejercicio.